In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
# Import necessary libraries
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from google.colab import drive  # For mounting Google Drive
import os  # For file path handling

# Mount Google Drive (assuming files are stored there; run this in Colab)
drive.mount('/content/drive')

# Define file paths (adjust 'MyDrive/path/to/files/' to your actual Drive path)
file1 = '/content/drive/MyDrive/Fault Localization/FCQ2E9B005 max data - 2025-12-11_2025-12-11.xls'
file2 = '/content/drive/MyDrive/Fault Localization/FCQ2E9B005 max data - 2025-12-17_2025-12-17.xls'

# Load the first dataset (skip initial rows if needed; adjust based on file structure)
df1 = pd.read_excel(file1, engine='xlrd', skiprows=0, header=0)  # Use openpyxl for .xls; skiprows to reach data

# Load the second dataset similarly
df2 = pd.read_excel(file2, engine='xlrd', skiprows=0, header=0)

# Convert 'Time' to datetime for both and set as index
df1['Time'] = pd.to_datetime(df1['Time'])
df2['Time'] = pd.to_datetime(df2['Time'])
df1 = df1.set_index('Time').sort_index()
df2 = df2.set_index('Time').sort_index()

# Combine the two DataFrames (concat along rows)
df_combined = pd.concat([df1, df2])

# Select relevant features for PV fault localization (based on key columns for strings: voltages, currents, energy, power, temps)
features = [
    col for col in df_combined.columns if col.startswith('Vpv') or col.startswith('Ipv') or col.startswith('Epv') or
    col in ['Ppv(W)', 'Pac(W)', 'INVTemp(℃)', 'AMTemp1(℃)', 'AMTemp2(℃)']
]
df_combined = df_combined[features]

# Filter to daytime/active data only (where Pac(W) > 0, as faults are detectable during generation)
df_day = df_combined[df_combined['Pac(W)'] > 0].copy()

# Add derived features: voltage differences (mismatches for fault detection) and current ratios (normalized for imbalances)
for i in range(1, 17):  # Up to 16 strings
    other_vpvs = [f'Vpv{j}(V)' for j in range(1, 17) if j != i]
    if other_vpvs:
        df_day[f'Vpv_diff{i}'] = df_day[f'Vpv{i}(V)'] - df_day[other_vpvs].mean(axis=1)
    df_day[f'Ipv_ratio{i}'] = df_day[f'Ipv{i}(A)'] / df_day[f'Ipv{i}(A)'].mean() if df_day[f'Ipv{i}(A)'].mean() != 0 else 0

# Normalize the data using MinMaxScaler (scale features to [0,1] for ML model stability)
scaler = MinMaxScaler()
df_preprocessed = pd.DataFrame(scaler.fit_transform(df_day), columns=df_day.columns, index=df_day.index)

# Export the preprocessed combined DataFrame to Excel (can be downloaded/extracted)
output_path = '/content/drive/MyDrive/preprocessed_combined_pv_data.xlsx'
df_preprocessed.to_excel(output_path, index=True)  # Include index (Time) for reference

print(f"Preprocessed file saved to: {output_path}")
print(f"Shape of preprocessed data: {df_preprocessed.shape}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Preprocessed file saved to: /content/drive/MyDrive/preprocessed_combined_pv_data.xlsx
Shape of preprocessed data: (293, 103)


In [12]:
from google.colab import files
files.download('/content/drive/MyDrive/preprocessed_combined_pv_data.xlsx')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')
file2 = '/content/drive/MyDrive/Fault Localization/FCQ2E9B005 max data - 2025-12-17_2025-12-17.xls'
df2 = pd.read_excel(file2, engine='xlrd', skiprows=0, header=0)
df2.columns

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Index(['Serial number', 'Time', 'Status', 'EacToday(kWh)', 'EacTotal(kWh)',
       'Vpv1(V)', 'Vpv2(V)', 'Vpv3(V)', 'Vpv4(V)', 'Vpv5(V)',
       ...
       'Debug1', 'Debug2', 'Debug3', 'ReactPower(Var)', 'ReactPowerMax(Var)',
       'ReactPower_Total(kWh)', 'isAgain', 'AfciStatus', 'StrengthPv1',
       'StrengthPv2'],
      dtype='object', length=254)

In [15]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from google.colab import drive  # For mounting Google Drive
import os  # For file path handling

# Install xlrd if not installed (for .xls files)
!pip install xlrd

# Mount Google Drive (assuming files are stored there; run this in Colab)
drive.mount('/content/drive')

# Define file paths
file1 = '/content/drive/MyDrive/Fault Localization/FCQ2E9B005 max data - 2025-12-11_2025-12-11.xls'
file2 = '/content/drive/MyDrive/Fault Localization/FCQ2E9B005 max data - 2025-12-17_2025-12-17.xls'

# Load the datasets
df1 = pd.read_excel(file1, engine='xlrd', skiprows=0, header=0)
df2 = pd.read_excel(file2, engine='xlrd', skiprows=0, header=0)

# Convert 'Time' to datetime
df1['Time'] = pd.to_datetime(df1['Time'])
df2['Time'] = pd.to_datetime(df2['Time'])
df1 = df1.set_index('Time').sort_index()
df2 = df2.set_index('Time').sort_index()

# Combine DataFrames
df_combined = pd.concat([df1, df2])

# Select relevant features
features = [
    col for col in df_combined.columns if col.startswith('Vpv') or col.startswith('Ipv') or col.startswith('Epv') or
    col in ['Ppv(W)', 'Pac(W)', 'INVTemp(℃)', 'AMTemp1(℃)', 'AMTemp2(℃)']
]
df_combined = df_combined[features]

# Filter daytime data
df_day = df_combined[df_combined['Pac(W)'] > 0].copy()

# Add derived features
for i in range(1, 17):
    other_vpvs = [f'Vpv{j}(V)' for j in range(1, 17) if j != i]
    if other_vpvs:
        df_day[f'Vpv_diff{i}'] = df_day[f'Vpv{i}(V)'] - df_day[other_vpvs].mean(axis=1)
    df_day[f'Ipv_ratio{i}'] = df_day[f'Ipv{i}(A)'] / df_day[f'Ipv{i}(A)'].mean() if df_day[f'Ipv{i}(A)'].mean() != 0 else 0

# Function to add synthetic faults with balanced samples
def add_synthetic_faults(df, num_samples_per_fault=100):  # Increased from 50 to 100
    synthetic = []
    labels = []

    # Ensure we have enough samples for each fault
    fault_types = ['open', 'short', 'shadow', 'hotspot']

    for fault_type in fault_types:
        for _ in range(num_samples_per_fault):
            # Ensure we sample from available data
            if len(df) > 0:
                row = df.sample(1).iloc[0].copy()
                str_id = np.random.randint(1, 17)

                if fault_type == 'open':
                    row[f'Vpv{str_id}(V)'] = 0
                    row[f'Ipv{str_id}(A)'] = 0
                    if f'Epv{str_id}Today(kWh)' in row:
                        row[f'Epv{str_id}Today(kWh)'] = 0

                elif fault_type == 'short':
                    row[f'Vpv{str_id}(V)'] = row[f'Vpv{str_id}(V)'] * 0.1
                    row[f'Ipv{str_id}(A)'] = row[f'Ipv{str_id}(A)'] * 1.5
                    if f'Epv{str_id}Today(kWh)' in row:
                        row[f'Epv{str_id}Today(kWh)'] *= 0.5

                elif fault_type == 'shadow':
                    reduction = np.random.uniform(0.3, 0.7)
                    row[f'Ipv{str_id}(A)'] *= reduction
                    if f'Epv{str_id}Today(kWh)' in row:
                        row[f'Epv{str_id}Today(kWh)'] *= reduction

                elif fault_type == 'hotspot':
                    if 'INVTemp(℃)' in row:
                        row['INVTemp(℃)'] = row['INVTemp(℃)'] * 1.2
                    row[f'Vpv{str_id}(V)'] *= 0.9
                    if f'Epv{str_id}Today(kWh)' in row:
                        row[f'Epv{str_id}Today(kWh)'] *= 0.9

                synthetic.append(row)
                labels.append(fault_type)  # Simplified label to just fault type

    syn_df = pd.DataFrame(synthetic, columns=df.columns)
    syn_df['label'] = labels

    # Add normal labels to original
    df['label'] = 'normal'

    # Combine
    combined = pd.concat([df, syn_df]).reset_index(drop=True)
    return combined

# Add synthetic faults
df_aug = add_synthetic_faults(df_day, num_samples_per_fault=100)

# Check class distribution
print("Class distribution in augmented data:")
print(df_aug['label'].value_counts())

# Ensure each class has at least 2 samples
class_counts = df_aug['label'].value_counts()
valid_classes = class_counts[class_counts >= 2].index.tolist()

if len(valid_classes) < len(class_counts):
    print(f"\nRemoving classes with only 1 sample. Keeping: {valid_classes}")
    df_aug = df_aug[df_aug['label'].isin(valid_classes)]

print(f"\nAugmented shape: {df_aug.shape}")

# Normalize data
features_to_scale = [col for col in df_aug.columns if col != 'label']
scaler = MinMaxScaler()
df_aug[features_to_scale] = scaler.fit_transform(df_aug[features_to_scale])

# Split data - WITHOUT stratify if still having issues
print("\nDataset small: Using train/test split only (80/20)")
try:
    df_train, df_test = train_test_split(df_aug, test_size=0.2, random_state=42, stratify=df_aug['label'])
    print("Stratified split successful")
except:
    print("Stratified split failed. Using simple random split.")
    df_train, df_test = train_test_split(df_aug, test_size=0.2, random_state=42)

df_valid = None

# Export splits
output_dir = '/content/drive/MyDrive/Fault Localization/'
df_aug.to_excel(os.path.join(output_dir, 'preprocessed_combined_pv_data.xlsx'), index=True)
df_train.to_excel(os.path.join(output_dir, 'train_pv_data.xlsx'), index=True)
df_test.to_excel(os.path.join(output_dir, 'test_pv_data.xlsx'), index=True)

print(f"\nFiles saved to: {output_dir}")
print(f"Train shape: {df_train.shape}, Test shape: {df_test.shape}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Class distribution in augmented data:
label
normal     293
open       100
short      100
shadow     100
hotspot    100
Name: count, dtype: int64

Augmented shape: (693, 104)

Dataset small: Using train/test split only (80/20)
Stratified split successful

Files saved to: /content/drive/MyDrive/Fault Localization/
Train shape: (554, 104), Test shape: (139, 104)
